# Phase 6.5 shard 09 (core65)

Runs **29 cells** of the frozen Phase 6.5 manifest (`G3-PHASE65-v1`), covering: `cwdb_dr`, `cwdb_r3_cvridge`, `cwdb_zipt`.

This shard runs the pure-Python methods: the cross-fitted R3 booster, the doubly-robust calibration layer, and the two-part assembly. There is no install step, so computing starts immediately.

Estimated single-threaded compute on the reference machine is about **83 minutes**. Colab cores are slower, so allow two to three times that, plus any install time above. This fits comfortably inside a nine hour session.

**Run every cell in order.** The last cell downloads a `.zip`; collect every shard's zip into `results/phase65/colab_shards/` (logs into `results/manifests/`) and run `python research/run_phase65.py merge`.


In [ ]:
# Thread pinning MUST happen before NumPy or SciPy are imported.
# OpenMP sizes its pool at initialisation, so setting these
# afterwards is silently ineffective.
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[_v] = '1'
print('threads pinned to 1')

## 1. Clone the repository at the pinned commit

Remote `https://github.com/hugogobato/wasserstein-causal-forests.git`, commit `bfe99cb3f305`. After checkout the notebook asserts the frozen manifest checksum, so a clone of anything but the generating commit fails here rather than mid-run.

In [ ]:
import subprocess, pathlib, os, sys, json, hashlib

REPO = 'https://github.com/hugogobato/wasserstein-causal-forests.git'
COMMIT = 'bfe99cb3f305d5f9372dfb723888ed128b8f6ed9'
EXPECTED_CHECKSUM = '4e28d308ca99cde4c81379524fc4492a15b38f029b449899b0a307b6c0ace110'

workdir = pathlib.Path('/content/wcf')
if not workdir.exists():
    subprocess.run(['git', 'init', '-q', str(workdir)], check=True)
    subprocess.run(
        ['git', '-C', str(workdir), 'remote', 'add', 'origin', REPO],
        check=True,
    )
# A shallow fetch of the exact commit: nothing else is downloaded.
    subprocess.run(
        ['git', '-C', str(workdir), 'fetch', '-q', '--depth', '1',
         'origin', COMMIT], check=True,
    )
    subprocess.run(
        ['git', '-C', str(workdir), 'checkout', '-q', 'FETCH_HEAD'],
        check=True,
    )
os.chdir(workdir)
sys.path.insert(0, str(workdir / 'src'))

manifest = json.load(open(
    'results/manifests/phase65_manifest.json', encoding='utf-8'
))
checksum = hashlib.sha256(
    json.dumps(manifest['cells'], sort_keys=True).encode('utf-8')
).hexdigest()
assert checksum == EXPECTED_CHECKSUM, (
    'the cloned manifest does not match the frozen grid: '
    f'{checksum} != {EXPECTED_CHECKSUM}'
)
print('repo ready at commit ' + COMMIT[:12] + '; '
      + str(manifest['n_cells']) + ' frozen cells verified')

## 2. Dependencies

In [ ]:
# This group needs only NumPy, SciPy, scikit-learn and PyArrow, all
# preinstalled on Colab. Nothing to install.
!python -c "import numpy, scipy, sklearn, pyarrow; print('numpy', numpy.__version__, '| sklearn', sklearn.__version__)"


## 3. This shard's cells

In [ ]:
import json, collections
SHARD_INDEX = 9
CELLS = json.loads('''[{"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 4, "cell_key": "3c27f4a1561dba68", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 3, "cell_key": "ec6c844987ce6dc3", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 9, "cell_key": "cc92db3b0ff1acd9", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 1, "cell_key": "d1265593127af69a", "test_seed": 900001}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 3, "cell_key": "3578187debbfc7c2", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 5, "cell_key": "68e940ffcf2f1fa7", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 7, "cell_key": "7f32513aaf1962eb", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 9, "cell_key": "fca19b639ee53017", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 1, "cell_key": "47f47ed141a36898", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 3, "cell_key": "bb180a89b7887802", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 5, "cell_key": "f43f555c3e1d22de", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 7, "cell_key": "f0f5b2fde484170a", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 9, "cell_key": "25bf7807a8b40ae6", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 3, "cell_key": "9e9b33eb15b311cd", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 5, "cell_key": "b01cc7bafc6a4308", "test_seed": 900005}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 7, "cell_key": "cc9e990cc821ff06", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 9, "cell_key": "bdf0bbf665a485b6", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 1, "cell_key": "a27539902e737624", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 3, "cell_key": "385759f825cb4167", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 5, "cell_key": "9aa260df7324daa9", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 7, "cell_key": "8f5d2f872ad177d6", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 9, "cell_key": "e81da2adbd1d64f5", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 1, "cell_key": "6d2b6a9221e79cf9", "test_seed": 900001}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 3, "cell_key": "2e71e9f0648b423d", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 5, "cell_key": "769b9f687c293cd1", "test_seed": 900005}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 7, "cell_key": "5c80e321ddb48b4c", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 9, "cell_key": "5548bb183129e03b", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 7, "cell_key": "03d96c5884460dfb", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_r3_cvridge", "seed": 9, "cell_key": "fcaed2228101b3b6", "test_seed": 900009}]''')
print(f'{len(CELLS)} cells in this shard')
for key, count in sorted(collections.Counter(
        (c['grid'], c['dgp'], c['method'])
        for c in CELLS).items()):
    print(f'  {key[0]:12s} {key[1]:8s} {key[2]:18s} {count}')

## 4. Run

In [ ]:
import time
from pathlib import Path
from wasserstein_causal_forests.g3.manifest import Cell
from wasserstein_causal_forests.g3.runner import run_shard

cells = [Cell(**{k: v for k, v in item.items()
                 if k not in ('cell_key', 'test_seed')})
         for item in CELLS]

out = Path('/content/wcf/results/phase65/colab_shards')
out.mkdir(parents=True, exist_ok=True)
log = Path(f'/content/wcf/results/manifests/phase65_execution_log_{SHARD_INDEX:03d}.jsonl')
log.parent.mkdir(parents=True, exist_ok=True)
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)

started = time.time()
summary = run_shard(
    cells,
    out / f'shard_{SHARD_INDEX:03d}.parquet',
    cache_directory=cache,
    log_path=log,
    manifest_contract_id='G3-PHASE65-v1',
)
print(json.dumps(summary, indent=2))
print(f'elapsed {(time.time() - started) / 60:.1f} min')

## Check

Every cell must appear exactly once, as a success or as a failure. Failures are kept and reported at merge time; a seed is never silently replaced.

In [ ]:
import collections
records = [json.loads(line) for line in
           open(log, encoding='utf-8') if line.strip()]
status = collections.Counter(r['status'] for r in records)
print('cells logged:', len(records), '| expected:', len(CELLS))
print('status:', dict(status))
assert len(records) == len(CELLS), 'shard did not finish every cell'
for record in records:
    if record['status'] != 'ok':
        print('  FAILED', record['dgp'], record['method'],
              record['seed'])
slowest = sorted(records, key=lambda r: -r['wall_seconds'])[:5]
print('slowest cells:', [(r['method'], round(r['wall_seconds'], 1))
                         for r in slowest])

## Download the results

In [ ]:
import shutil
bundle = '/content/p65_shard_09_core65'
staging = Path('/content/bundle')
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)
shutil.copy(out / f'shard_{SHARD_INDEX:03d}.parquet', staging)
if log.exists():
    shutil.copy(log, staging)
output_file = shutil.make_archive(bundle, 'zip', staging)
print('bundle:', output_file,
      f'({os.path.getsize(output_file) / 1e6:.2f} MB)')

try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)